# 23CSE301 — ML Capstone — Review 1
## Track: Gig Economy / Rider–Driver Earnings & Burnout
### Dataset: `chicago_taxi_sample.csv`

**Regression target (numeric):** `total_earnings` — a driver's total earnings for a given day, aggregated from trip-level fares/tips/tolls/extras (`trip_total`), predicted from workload, behaviour, and categorical features.

**Classification target (categorical, Part A this review):** `burnout_risk` ∈ {Low, Medium, High} — binned from a driver's total hours driven that day, predicted from the same features (hours-driven itself is **excluded** to avoid label leakage).

**Primary tools used:** Python 3, scikit-learn, Pandas, NumPy, Matplotlib, Seaborn — all imported and used directly below (no other modelling library is used, per the guidelines).

> Wherever you see a `**Observation:**` markdown cell, run the cell(s) above it first, then fill in one or two sentences based on the printed numbers/plot — this satisfies rubric item **A3 (Insight commentary)**, which requires your own written interpretation, not a generic one.

Random seed is fixed to `42` everywhere for reproducibility, per the guidelines.

### Team ownership for this review (as assigned)

| Member | Responsible for |
|---|---|
| **Member 1** | Regression algorithms 1–5: Linear, Ridge, Lasso, ElasticNet, Polynomial Regression |
| **Member 2** | Regression algorithms 6–10: Decision Tree, Random Forest, Gradient Boosting, SVR, KNN Regressor |
| **Member 3** | Classification Track Part A (all 5): Logistic Regression, KNN, Naive Bayes, Decision Tree, SVM |

Sections A (EDA) and B (preprocessing) are shared/setup work all three members should understand — the viva can ask any member about any cell, per guideline 7.4. Each algorithm cell below is tagged `# Owner: Member N` so it's obvious in the notebook (and should match your Git commit authorship for the D2/GitHub criterion).

In [ ]:
import warnings
warnings.filterwarnings("ignore")

# Primary tools (per guidelines section 1): Python 3, scikit-learn, Pandas, NumPy, Matplotlib, Seaborn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, OneHotEncoder

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR, SVC
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error,
                              accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, ConfusionMatrixDisplay)

sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams["figure.dpi"] = 100
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Section A — Dataset & EDA

### A1. Dataset loading & audit (shape, dtypes, missing values, target distribution)

In [ ]:
DATA_PATH = "chicago_taxi_sample.csv"
df_raw = pd.read_csv(DATA_PATH)

print("Shape:", df_raw.shape)
print("\nDtypes:\n", df_raw.dtypes)
print("\nMissing values per column:\n", df_raw.isna().sum().sort_values(ascending=False))
print("\nDescribe (numeric):\n", df_raw.describe())

print("\n--- Categorical / target-relevant distributions ---")
print("\npayment_type value counts:\n", df_raw["payment_type"].value_counts(dropna=False))
print("\ncompany — unique count:", df_raw["company"].nunique(), " | top 5:\n", df_raw["company"].value_counts().head())
print("\npickup_community_area — unique count:", df_raw["pickup_community_area"].nunique())
print("\ndropoff_community_area — unique count:", df_raw["dropoff_community_area"].nunique())
print("\ntrip_total (the raw quantity our regression target is built from) — describe:\n", df_raw["trip_total"].describe())

**Observation:** _[Fill in after running: comment on dataset size, which columns have the most missing values, how many distinct payment types/companies/community areas exist, and whether `trip_total` looks skewed based on the describe() output.]_

### A2. EDA visualisations — numeric feature distributions

In [ ]:
numeric_cols = ["trip_seconds", "trip_miles", "fare", "tips", "tolls", "extras", "trip_total"]

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    sns.histplot(df_raw[col].dropna(), bins=50, kde=True, ax=axes[i])
    axes[i].set_title(f"Distribution of {col}")
    axes[i].set_xlabel(col)
for j in range(len(numeric_cols), len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.savefig("eda_feature_distributions.png", bbox_inches="tight")
plt.show()

for col in numeric_cols:
    print(f"{col}: skew={df_raw[col].skew():.2f}")

**Observation:** _[Fill in: which features are heavily right-skewed (e.g. trip_total, tips)? What does that imply about outliers/whether IQR clipping is warranted?]_

### A2 (cont). Categorical feature distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

df_raw["payment_type"].value_counts().plot(kind="bar", ax=axes[0], color="slateblue")
axes[0].set_title("payment_type distribution")
axes[0].set_xlabel("payment_type")
axes[0].set_ylabel("count")
axes[0].tick_params(axis="x", rotation=45)

df_raw["company"].value_counts().head(10).plot(kind="bar", ax=axes[1], color="darkcyan")
axes[1].set_title("Top 10 company distribution")
axes[1].set_xlabel("company")
axes[1].set_ylabel("count")
axes[1].tick_params(axis="x", rotation=75)

df_raw["pickup_community_area"].value_counts().head(15).plot(kind="bar", ax=axes[2], color="salmon")
axes[2].set_title("Top 15 pickup_community_area distribution")
axes[2].set_xlabel("pickup_community_area")
axes[2].set_ylabel("count")

plt.tight_layout()
plt.savefig("eda_categorical_distributions.png", bbox_inches="tight")
plt.show()

**Observation:** _[Fill in: is one payment type (e.g. cash/card) dominant? Are trips concentrated in a handful of companies or community areas, or spread evenly? This matters for whether categorical encodings will have very rare categories.]_

### A2 (cont). Temporal distribution — pickup hour of day

In [ ]:
_ts_preview = pd.to_datetime(df_raw["trip_start_timestamp"], errors="coerce")
plt.figure(figsize=(9, 5))
sns.histplot(_ts_preview.dt.hour.dropna(), bins=24, color="mediumseagreen")
plt.title("Distribution of trips by pickup hour of day")
plt.xlabel("Hour of day (0-23)")
plt.ylabel("Number of trips")
plt.tight_layout()
plt.savefig("eda_pickup_hour_distribution.png", bbox_inches="tight")
plt.show()

**Observation:** _[Fill in: are there clear peak hours (e.g. morning/evening rush, or late-night)? This directly motivates the `is_night`/`pickup_hour` engineered features used later.]_

### A2 (cont). Correlation heatmap

In [ ]:
plt.figure(figsize=(9, 7))
corr = df_raw[numeric_cols].corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Correlation Heatmap — Numeric Trip Features")
plt.tight_layout()
plt.savefig("eda_correlation_heatmap.png", bbox_inches="tight")
plt.show()

**Observation:** _[Fill in: which pairs are most correlated with `trip_total`? Does `fare` dominate, and is that expected given trip_total = fare + tips + tolls + extras?]_

### A2 (cont). Target distribution — `trip_total`

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df_raw["trip_total"].clip(upper=df_raw["trip_total"].quantile(0.99)), bins=60, kde=True, color="teal")
plt.title("Distribution of trip_total (clipped at 99th percentile)")
plt.xlabel("trip_total ($)")
plt.tight_layout()
plt.savefig("eda_target_distribution.png", bbox_inches="tight")
plt.show()

**Observation:** _[Fill in: where is the bulk of trips concentrated in $? Is there a long right tail of high-fare trips?]_

### A2 (cont). Scatter plots — feature vs. target relationships

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(data=df_raw.sample(min(5000, len(df_raw)), random_state=RANDOM_STATE),
                 x="trip_miles", y="trip_total", alpha=0.4, ax=axes[0])
axes[0].set_title("trip_miles vs trip_total")

sns.scatterplot(data=df_raw.sample(min(5000, len(df_raw)), random_state=RANDOM_STATE),
                 x="trip_seconds", y="trip_total", alpha=0.4, ax=axes[1], color="darkorange")
axes[1].set_title("trip_seconds vs trip_total")

plt.tight_layout()
plt.savefig("eda_scatter_feature_target.png", bbox_inches="tight")
plt.show()

**Observation:** _[Fill in: is the relationship between distance/duration and earnings roughly linear? Any visible outliers — e.g. very long trips with near-zero fare, or very short trips with high fare?]_

## Section B — Preprocessing & Feature Engineering

### B1. Data cleaning — missing values, duplicates, outliers

In [ ]:
df = df_raw.copy()

# Duplicates
n_dupes = df.duplicated(subset=["unique_key"]).sum()
print("Duplicate unique_key rows:", n_dupes)
df = df.drop_duplicates(subset=["unique_key"])

# Missing-value strategy: rows with a missing target or missing core trip fields are dropped
# (imputing a missing target/duration for a taxi trip would fabricate the label / workload signal).
core_cols = ["trip_start_timestamp", "trip_end_timestamp", "trip_seconds", "trip_miles",
             "trip_total", "fare", "pickup_community_area", "dropoff_community_area", "taxi_id"]
before = len(df)
df = df.dropna(subset=core_cols)
print(f"Dropped {before - len(df)} rows with missing core fields (kept {len(df)}).")

# Remaining, non-core missing values (e.g. company) are imputed with an explicit 'Unknown' category
df["company"] = df["company"].fillna("Unknown")
df["payment_type"] = df["payment_type"].fillna("Unknown")

# Outlier treatment: physically implausible trips (zero/negative duration or distance,
# unrealistic speed) are removed rather than clipped, since they represent logging errors,
# not genuine extreme-but-valid trips.
df = df[(df["trip_seconds"] > 0) & (df["trip_miles"] >= 0) & (df["trip_total"] >= 0)]

# Cap extreme outliers using IQR-based clipping (3x IQR) rather than dropping, to preserve sample size
for col in ["trip_total", "trip_miles", "trip_seconds"]:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    upper = q3 + 3 * iqr
    df[col] = df[col].clip(upper=upper)

print("Shape after cleaning:", df.shape)

**B1 justification:** rows missing core identifiers/target are dropped (imputing a trip's duration or fare would fabricate the label); `company`/`payment_type` are imputed with an explicit `'Unknown'` category since missingness there is plausibly informative (e.g. independent/unlicensed operators) rather than random. Physically impossible trips (zero duration, negative distance) are removed as logging errors; remaining extreme-but-valid values are IQR-capped (3×IQR) rather than dropped, to retain sample size for the ensemble/boosting models.

### B3. Feature engineering (trip-level)

In [ ]:
df["trip_start_timestamp"] = pd.to_datetime(df["trip_start_timestamp"], errors="coerce")
df["trip_end_timestamp"] = pd.to_datetime(df["trip_end_timestamp"], errors="coerce")
df = df.dropna(subset=["trip_start_timestamp"])

df["pickup_hour"] = df["trip_start_timestamp"].dt.hour
df["pickup_dayofweek"] = df["trip_start_timestamp"].dt.dayofweek  # 0=Mon
df["trip_date"] = df["trip_start_timestamp"].dt.date
df["is_weekend"] = df["pickup_dayofweek"].isin([5, 6]).astype(int)
df["is_night"] = df["pickup_hour"].apply(lambda h: 1 if (h >= 22 or h < 5) else 0)

# trip_speed_mph: engineered feature — a driver's effective earning rate depends not just
# on distance but on speed (congestion vs open road), which plausibly predicts fare/time efficiency
# and is a workload-intensity signal relevant to burnout.
df["trip_speed_mph"] = df["trip_miles"] / (df["trip_seconds"] / 3600)
df["trip_speed_mph"] = df["trip_speed_mph"].replace([np.inf, -np.inf], np.nan)
df["trip_speed_mph"] = df["trip_speed_mph"].fillna(df["trip_speed_mph"].median())
df["trip_speed_mph"] = df["trip_speed_mph"].clip(upper=80)  # cap implausible speeds

print(df[["pickup_hour", "is_weekend", "is_night", "trip_speed_mph"]].describe())

**B3 justification:** `trip_speed_mph` is engineered because raw distance/duration alone don't capture *efficiency* — two trips of equal duration but very different speeds imply very different traffic/workload conditions, which is directly relevant to a burnout framing. `pickup_hour`, `is_night`, and `is_weekend` are engineered because gig-driver fatigue and earnings both vary strongly by time-of-day/week (night and weekend shifts are widely reported as higher-strain, and often higher-earning).

### B2 / B3 (cont). Driver-day aggregation — building the two modelling targets, including a categorical feature

In [ ]:
# Mode helper for the dominant category of a driver's day (used to build a categorical feature)
def _mode_or_unknown(s):
    m = s.mode()
    return m.iloc[0] if len(m) else "Unknown"

agg_full = df.groupby(["taxi_id", "trip_date"]).agg(
    trip_count=("unique_key", "count"),
    total_miles=("trip_miles", "sum"),
    total_duration_sec=("trip_seconds", "sum"),
    avg_speed_mph=("trip_speed_mph", "mean"),
    pct_night_trips=("is_night", "mean"),
    pct_weekend_trips=("is_weekend", "mean"),
    num_unique_pickup_areas=("pickup_community_area", "nunique"),
    num_unique_dropoff_areas=("dropoff_community_area", "nunique"),
    total_earnings=("trip_total", "sum"),
    dominant_payment_type=("payment_type", _mode_or_unknown),  # categorical feature -> will be one-hot encoded
).reset_index()

# Require at least 2 trips in a driver-day so aggregated features are meaningful, not single-trip noise.
# chicago_taxi_sample.csv is a SAMPLE file, so this can leave very few driver-days — if so, relax to >=1.
MIN_TRIPS_PER_DRIVER_DAY = 2
agg = agg_full[agg_full["trip_count"] >= MIN_TRIPS_PER_DRIVER_DAY].reset_index(drop=True)

if len(agg) < 30:
    print(f"Only {len(agg)} driver-days had >= {MIN_TRIPS_PER_DRIVER_DAY} trips — too few to model reliably "
          f"(expected on a *sample* file). Relaxing the minimum to >=1 trip/driver-day.")
    MIN_TRIPS_PER_DRIVER_DAY = 1
    agg = agg_full[agg_full["trip_count"] >= MIN_TRIPS_PER_DRIVER_DAY].reset_index(drop=True)

agg["total_hours"] = agg["total_duration_sec"] / 3600

# Classification target: burnout_risk tier.
# NOTE: fixed real-world hour thresholds (e.g. <4h/4-8h/>8h) are NOT used here, because on a *sample*
# dataset the actual hours-driven distribution may not span those ranges at all (e.g. almost every
# driver-day could fall under 4h), which would leave one or more classes empty or with a single
# member and break model fitting. Quantile-based binning instead guarantees roughly balanced classes
# no matter how the sample is distributed. `.rank(method="first")` breaks ties so qcut can always
# produce 3 bins even when many driver-days have identical/near-identical total_hours.
agg["burnout_risk"] = pd.qcut(agg["total_hours"].rank(method="first"), q=3, labels=["Low", "Medium", "High"])

# Coarser payment-type grouping to avoid a very-rare-category column exploding the one-hot encoding
top_payment_types = agg["dominant_payment_type"].value_counts().head(3).index
agg["dominant_payment_type"] = agg["dominant_payment_type"].where(
    agg["dominant_payment_type"].isin(top_payment_types), "Other"
)

print("Driver-day rows:", agg.shape)
print("\nburnout_risk class distribution:\n", agg["burnout_risk"].value_counts())
print("\ndominant_payment_type distribution:\n", agg["dominant_payment_type"].value_counts())

n_classes = agg["burnout_risk"].nunique()
if n_classes < 2 or agg["burnout_risk"].value_counts().min() < 2:
    raise ValueError(
        f"burnout_risk has only {n_classes} usable class(es) even after quantile binning — "
        f"the sample is too small/homogeneous for driver-day classification. Check df.shape and "
        f"agg_full.shape above: if agg_full has very few rows, chicago_taxi_sample.csv itself may "
        f"only cover a handful of taxi_id/day combinations, and a trip-level target (rather than "
        f"driver-day aggregation) would be a better fit for this sample size."
    )

agg.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.countplot(data=agg, x="burnout_risk", order=["Low", "Medium", "High"], ax=axes[0])
axes[0].set_title("Burnout-risk class distribution (driver-days)")
axes[0].set_xlabel("Burnout risk tier")
axes[0].set_ylabel("Number of driver-days")

sns.countplot(data=agg, x="dominant_payment_type", ax=axes[1], color="goldenrod")
axes[1].set_title("Dominant payment type per driver-day")
axes[1].set_xlabel("dominant_payment_type")

plt.tight_layout()
plt.savefig("eda_burnout_and_payment_distribution.png", bbox_inches="tight")
plt.show()

**Observation:** _[Fill in: is the class distribution balanced across Low/Medium/High? If one class dominates, note that this motivates using weighted F1 rather than raw accuracy for evaluation. Is one payment type overwhelmingly dominant among driver-days?]_

### B2. Encoding, scaling & train/test split (stratified for both tracks)

In [ ]:
# ---- Regression setup ----
reg_numeric_features = ["trip_count", "total_miles", "avg_speed_mph", "pct_night_trips",
                         "pct_weekend_trips", "num_unique_pickup_areas", "num_unique_dropoff_areas"]
reg_categorical_features = ["dominant_payment_type"]

X_reg_raw = agg[reg_numeric_features + reg_categorical_features].copy()
y_reg = agg["total_earnings"].copy()

# Stratify the regression split on a binned version of the (continuous) target, so the train/test
# split is representative across low/mid/high earners, not just a random 80:20 split.
# Falls back to a plain (non-stratified) split if the sample is too small for 5 quantile bins.
try:
    earnings_bins = pd.qcut(y_reg.rank(method="first"), q=5)
    X_reg_train_raw, X_reg_test_raw, y_reg_train, y_reg_test = train_test_split(
        X_reg_raw, y_reg, test_size=0.2, random_state=RANDOM_STATE, stratify=earnings_bins
    )
except ValueError as e:
    print(f"Stratified regression split failed ({e}) — likely too few rows for 5 quantile bins "
          f"on this sample. Falling back to a plain random 80:20 split.")
    X_reg_train_raw, X_reg_test_raw, y_reg_train, y_reg_test = train_test_split(
        X_reg_raw, y_reg, test_size=0.2, random_state=RANDOM_STATE
    )

# Encoding: one-hot encode the categorical feature, FIT on train only, transform both splits
ohe_reg = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
reg_cat_train = ohe_reg.fit_transform(X_reg_train_raw[reg_categorical_features])
reg_cat_test = ohe_reg.transform(X_reg_test_raw[reg_categorical_features])
reg_cat_cols = ohe_reg.get_feature_names_out(reg_categorical_features)

# Scaling: numeric features scaled, FIT on train only, transform both splits (no data leakage)
scaler_reg = StandardScaler()
reg_num_train = scaler_reg.fit_transform(X_reg_train_raw[reg_numeric_features])
reg_num_test = scaler_reg.transform(X_reg_test_raw[reg_numeric_features])

reg_features = list(reg_numeric_features) + list(reg_cat_cols)
X_reg_train_s = pd.DataFrame(np.hstack([reg_num_train, reg_cat_train]), columns=reg_features, index=X_reg_train_raw.index)
X_reg_test_s = pd.DataFrame(np.hstack([reg_num_test, reg_cat_test]), columns=reg_features, index=X_reg_test_raw.index)

print("Regression train/test:", X_reg_train_s.shape, X_reg_test_s.shape)
print("Regression feature columns:", reg_features)

# ---- Classification setup (Part A) ----
clf_numeric_features = ["trip_count", "total_miles", "avg_speed_mph", "pct_night_trips",
                         "pct_weekend_trips", "num_unique_pickup_areas", "num_unique_dropoff_areas",
                         "total_earnings"]
clf_categorical_features = ["dominant_payment_type"]

X_clf_raw = agg[clf_numeric_features + clf_categorical_features].copy()
y_clf = agg["burnout_risk"].copy()

try:
    X_clf_train_raw, X_clf_test_raw, y_clf_train, y_clf_test = train_test_split(
        X_clf_raw, y_clf, test_size=0.2, random_state=RANDOM_STATE, stratify=y_clf
    )
except ValueError as e:
    print(f"Stratified classification split failed ({e}) — falling back to a plain random 80:20 split.")
    X_clf_train_raw, X_clf_test_raw, y_clf_train, y_clf_test = train_test_split(
        X_clf_raw, y_clf, test_size=0.2, random_state=RANDOM_STATE
    )

# Final safety check: fail loudly and clearly (not with sklearn's cryptic error) if a split still
# produced a single-class training set, which would make every classifier below un-fittable.
if y_clf_train.nunique() < 2:
    raise ValueError(
        "y_clf_train contains only one class after splitting — the sample is too small/imbalanced "
        "for this 3-class target even with quantile binning. Re-run the driver-day aggregation cell "
        "above and check agg['burnout_risk'].value_counts(); consider a 2-class target (median split) "
        "instead of 3 if the sample stays this small."
    )

ohe_clf = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
clf_cat_train = ohe_clf.fit_transform(X_clf_train_raw[clf_categorical_features])
clf_cat_test = ohe_clf.transform(X_clf_test_raw[clf_categorical_features])
clf_cat_cols = ohe_clf.get_feature_names_out(clf_categorical_features)

scaler_clf = StandardScaler()
clf_num_train = scaler_clf.fit_transform(X_clf_train_raw[clf_numeric_features])
clf_num_test = scaler_clf.transform(X_clf_test_raw[clf_numeric_features])

clf_features = list(clf_numeric_features) + list(clf_cat_cols)
X_clf_train_s = pd.DataFrame(np.hstack([clf_num_train, clf_cat_train]), columns=clf_features, index=X_clf_train_raw.index)
X_clf_test_s = pd.DataFrame(np.hstack([clf_num_test, clf_cat_test]), columns=clf_features, index=X_clf_test_raw.index)

print("\nClassification train/test:", X_clf_train_s.shape, X_clf_test_s.shape)
print("Classification feature columns:", clf_features)

**B2 note:** both the `OneHotEncoder` (for `dominant_payment_type`) and `StandardScaler` (for numeric features) are `fit` on the **training split only** and `transform`-applied to test — satisfying the no-leakage requirement for both encoding and scaling. The **classification** split is stratified on `burnout_risk` directly; the **regression** split is stratified on quintile-binned `total_earnings`, since `train_test_split` can only stratify on discrete labels, not a continuous target. Both stratified splits fall back to a plain random split (with a printed warning) if the sample is too small for the requested number of bins — this only matters on very small samples and does not change the modelling approach otherwise.

## Section C — Regression Track (10 algorithms)

Each algorithm below is trained and evaluated independently (no shared `Pipeline` object, per requirement), with its own predicted-vs-actual visualisation. Fitted models and predictions are kept so Section C4 can dynamically pick out whichever model is actually best on your data.

In [ ]:
reg_results = []
fitted_reg_models = {}  # name -> (model, preds) for later dynamic best-model lookup

def evaluate_regressor(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    r2 = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)

    reg_results.append({"Algorithm": name, "R2": r2, "RMSE": rmse, "MAE": mae})
    fitted_reg_models[name] = (model, preds)
    print(f"{name}: R2={r2:.3f}  RMSE={rmse:.2f}  MAE={mae:.2f}")

    plt.figure(figsize=(6, 5))
    plt.scatter(y_test, preds, alpha=0.4, color="steelblue")
    lims = [min(y_test.min(), preds.min()), max(y_test.max(), preds.max())]
    plt.plot(lims, lims, "r--", label="Perfect prediction")
    plt.xlabel("Actual total_earnings")
    plt.ylabel("Predicted total_earnings")
    plt.title(f"{name}: Predicted vs Actual")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"reg_{name.replace(' ', '_').lower()}_pred_vs_actual.png", bbox_inches="tight")
    plt.show()

    return model

### Member 1 — Regression algorithms 1–5

**1. Linear Regression** — baseline; interpret coefficients. `# Owner: Member 1`

In [ ]:
# Owner: Member 1
lin_reg = evaluate_regressor("Linear Regression", LinearRegression(),
                              X_reg_train_s, X_reg_test_s, y_reg_train, y_reg_test)

coef_table = pd.Series(lin_reg.coef_, index=reg_features).sort_values(key=abs, ascending=False)
print("\nStandardized coefficients (interpret relative to scaled features):\n", coef_table)

**2. Ridge Regression** — L2 regularisation; tune `alpha` (with an explicit before/after comparison). `# Owner: Member 1`

In [ ]:
# Owner: Member 1

# Baseline (untuned, default alpha=1.0) so we can report the improvement from tuning
ridge_baseline = Ridge(alpha=1.0, random_state=RANDOM_STATE).fit(X_reg_train_s, y_reg_train)
r2_ridge_baseline = r2_score(y_reg_test, ridge_baseline.predict(X_reg_test_s))

ridge_grid = GridSearchCV(Ridge(random_state=RANDOM_STATE), param_grid={"alpha": [0.01, 0.1, 1, 10, 100]},
                           cv=5, scoring="r2")
ridge_grid.fit(X_reg_train_s, y_reg_train)
print("Best Ridge alpha:", ridge_grid.best_params_)

ridge_best = evaluate_regressor("Ridge Regression", Ridge(alpha=ridge_grid.best_params_["alpha"], random_state=RANDOM_STATE),
                                 X_reg_train_s, X_reg_test_s, y_reg_train, y_reg_test)

r2_ridge_tuned = fitted_reg_models["Ridge Regression"][0].score(X_reg_test_s, y_reg_test)
print(f"Improvement from tuning: R2 {r2_ridge_baseline:.4f} (alpha=1.0, default) -> {r2_ridge_tuned:.4f} (alpha={ridge_grid.best_params_['alpha']}, tuned)")

**3. Lasso Regression** — L1 regularisation; observe feature sparsity. `# Owner: Member 1`

In [ ]:
# Owner: Member 1
lasso_grid = GridSearchCV(Lasso(random_state=RANDOM_STATE, max_iter=10000), param_grid={"alpha": [0.001, 0.01, 0.1, 1, 10]},
                           cv=5, scoring="r2")
lasso_grid.fit(X_reg_train_s, y_reg_train)
print("Best Lasso alpha:", lasso_grid.best_params_)
lasso_best = evaluate_regressor("Lasso Regression", Lasso(alpha=lasso_grid.best_params_["alpha"], random_state=RANDOM_STATE, max_iter=10000),
                                 X_reg_train_s, X_reg_test_s, y_reg_train, y_reg_test)

zeroed = np.array(reg_features)[np.isclose(lasso_best.coef_, 0)]
print("Features zeroed out by Lasso:", list(zeroed) if len(zeroed) else "None")

**4. ElasticNet Regression** — combined L1+L2; tune `l1_ratio`. `# Owner: Member 1`

In [ ]:
# Owner: Member 1
enet_grid = GridSearchCV(ElasticNet(random_state=RANDOM_STATE, max_iter=10000),
                          param_grid={"alpha": [0.01, 0.1, 1], "l1_ratio": [0.2, 0.5, 0.8]},
                          cv=5, scoring="r2")
enet_grid.fit(X_reg_train_s, y_reg_train)
print("Best ElasticNet params:", enet_grid.best_params_)
enet_best = evaluate_regressor("ElasticNet Regression",
                                ElasticNet(**enet_grid.best_params_, random_state=RANDOM_STATE, max_iter=10000),
                                X_reg_train_s, X_reg_test_s, y_reg_train, y_reg_test)

**5. Polynomial Regression** — `PolynomialFeatures` + Linear Regression; compare degrees. `# Owner: Member 1`

In [ ]:
# Owner: Member 1
for degree in [2, 3]:
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_train_poly = poly.fit_transform(X_reg_train_s)
    X_test_poly = poly.transform(X_reg_test_s)
    evaluate_regressor(f"Polynomial Regression (deg={degree})", LinearRegression(),
                        X_train_poly, X_test_poly, y_reg_train, y_reg_test)

### Member 2 — Regression algorithms 6–10

**6. Decision Tree Regressor** — tune `max_depth`; show feature importance. `# Owner: Member 2`

In [ ]:
# Owner: Member 2
dt_grid = GridSearchCV(DecisionTreeRegressor(random_state=RANDOM_STATE),
                        param_grid={"max_depth": [3, 5, 8, 12, None]}, cv=5, scoring="r2")
dt_grid.fit(X_reg_train_s, y_reg_train)
print("Best Decision Tree max_depth:", dt_grid.best_params_)
dt_best = evaluate_regressor("Decision Tree Regressor",
                              DecisionTreeRegressor(max_depth=dt_grid.best_params_["max_depth"], random_state=RANDOM_STATE),
                              X_reg_train_s, X_reg_test_s, y_reg_train, y_reg_test)

plt.figure(figsize=(7, 4))
pd.Series(dt_best.feature_importances_, index=reg_features).sort_values().plot(kind="barh", color="seagreen")
plt.title("Decision Tree Regressor — Feature Importance")
plt.tight_layout()
plt.savefig("reg_decision_tree_feature_importance.png", bbox_inches="tight")
plt.show()

**7. Random Forest Regressor** — ensemble baseline; tune `n_estimators` (with before/after comparison). `# Owner: Member 2`

In [ ]:
# Owner: Member 2

# Baseline (untuned, default n_estimators=100)
rf_baseline = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE).fit(X_reg_train_s, y_reg_train)
r2_rf_baseline = r2_score(y_reg_test, rf_baseline.predict(X_reg_test_s))

rf_grid = GridSearchCV(RandomForestRegressor(random_state=RANDOM_STATE),
                        param_grid={"n_estimators": [100, 200, 400]}, cv=5, scoring="r2")
rf_grid.fit(X_reg_train_s, y_reg_train)
print("Best Random Forest n_estimators:", rf_grid.best_params_)
rf_best = evaluate_regressor("Random Forest Regressor",
                              RandomForestRegressor(n_estimators=rf_grid.best_params_["n_estimators"], random_state=RANDOM_STATE),
                              X_reg_train_s, X_reg_test_s, y_reg_train, y_reg_test)

r2_rf_tuned = fitted_reg_models["Random Forest Regressor"][0].score(X_reg_test_s, y_reg_test)
print(f"Improvement from tuning: R2 {r2_rf_baseline:.4f} (n_estimators=100, default) -> {r2_rf_tuned:.4f} (n_estimators={rf_grid.best_params_['n_estimators']}, tuned)")

**8. Gradient Boosting Regressor** — tune `learning_rate`. `# Owner: Member 2`

In [ ]:
# Owner: Member 2
gb_grid = GridSearchCV(GradientBoostingRegressor(random_state=RANDOM_STATE),
                        param_grid={"learning_rate": [0.01, 0.05, 0.1, 0.2]}, cv=5, scoring="r2")
gb_grid.fit(X_reg_train_s, y_reg_train)
print("Best Gradient Boosting learning_rate:", gb_grid.best_params_)
gb_best = evaluate_regressor("Gradient Boosting Regressor",
                              GradientBoostingRegressor(learning_rate=gb_grid.best_params_["learning_rate"], random_state=RANDOM_STATE),
                              X_reg_train_s, X_reg_test_s, y_reg_train, y_reg_test)

**9. Support Vector Regressor (SVR)** — features already scaled; tune `C` and `kernel`. `# Owner: Member 2`

In [ ]:
# Owner: Member 2
svr_grid = GridSearchCV(SVR(), param_grid={"C": [1, 10, 100], "kernel": ["linear", "rbf"]}, cv=5, scoring="r2")
svr_grid.fit(X_reg_train_s, y_reg_train)
print("Best SVR params:", svr_grid.best_params_)
svr_best = evaluate_regressor("Support Vector Regressor",
                               SVR(**svr_grid.best_params_),
                               X_reg_train_s, X_reg_test_s, y_reg_train, y_reg_test)

**10. K-Nearest Neighbors Regressor** — tune `k`; scaling matters here since KNN is distance-based. `# Owner: Member 2`

In [ ]:
# Owner: Member 2
knn_grid = GridSearchCV(KNeighborsRegressor(), param_grid={"n_neighbors": [3, 5, 7, 9, 15]}, cv=5, scoring="r2")
knn_grid.fit(X_reg_train_s, y_reg_train)
print("Best KNN k:", knn_grid.best_params_)
knn_best = evaluate_regressor("KNN Regressor",
                               KNeighborsRegressor(n_neighbors=knn_grid.best_params_["n_neighbors"]),
                               X_reg_train_s, X_reg_test_s, y_reg_train, y_reg_test)

### C2. Comparative evaluation — all 10 regression models

In [ ]:
reg_comparison = pd.DataFrame(reg_results).sort_values("R2", ascending=False).reset_index(drop=True)
print(reg_comparison.to_string(index=False))
reg_comparison

### C3. 5-fold cross-validated R² for the two best-performing models

In [ ]:
top2_names = reg_comparison.iloc[:2]["Algorithm"].tolist()
print("Top 2 models by test R2:", top2_names)

model_lookup = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=ridge_grid.best_params_["alpha"], random_state=RANDOM_STATE),
    "Lasso Regression": Lasso(alpha=lasso_grid.best_params_["alpha"], random_state=RANDOM_STATE, max_iter=10000),
    "ElasticNet Regression": ElasticNet(**enet_grid.best_params_, random_state=RANDOM_STATE, max_iter=10000),
    "Decision Tree Regressor": DecisionTreeRegressor(max_depth=dt_grid.best_params_["max_depth"], random_state=RANDOM_STATE),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=rf_grid.best_params_["n_estimators"], random_state=RANDOM_STATE),
    "Gradient Boosting Regressor": GradientBoostingRegressor(learning_rate=gb_grid.best_params_["learning_rate"], random_state=RANDOM_STATE),
    "Support Vector Regressor": SVR(**svr_grid.best_params_),
    "KNN Regressor": KNeighborsRegressor(n_neighbors=knn_grid.best_params_["n_neighbors"]),
}

for name in top2_names:
    if name in model_lookup:
        scores = cross_val_score(model_lookup[name], X_reg_train_s, y_reg_train, cv=5, scoring="r2")
        print(f"{name}: 5-fold CV R2 = {scores.mean():.3f} (+/- {scores.std():.3f})")
    else:
        print(f"{name}: a Polynomial-Regression variant won — refit LinearRegression on the same "
              f"degree's PolynomialFeatures-expanded X_reg_train_s to 5-fold CV it if selected.")

### C4. Residual plot, predicted-vs-actual for the ACTUAL best model, and feature importance

In [ ]:
best_name = reg_comparison.iloc[0]["Algorithm"]
print("Best regression model (by test R2):", best_name)

# Dynamically pull the actual best model's fitted object + predictions (not hardcoded),
# so this cell is correct no matter which of the 10 algorithms wins on your real data.
if best_name in fitted_reg_models:
    best_model, best_preds = fitted_reg_models[best_name]
else:
    # Polynomial-degree variants aren't stored as a plain estimator; fall back to Random Forest,
    # which is always available and tree-based (also satisfies the feature-importance requirement).
    print(f"Note: '{best_name}' has no plain estimator object to introspect directly (e.g. Polynomial "
          f"Regression uses a transformed feature space) — using Random Forest Regressor for the "
          f"residual/importance plots below instead. Swap this in for your actual best model if it differs.")
    best_model, best_preds = fitted_reg_models["Random Forest Regressor"]
    best_name = "Random Forest Regressor"

residuals = y_reg_test - best_preds

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(best_preds, residuals, alpha=0.4, color="crimson")
axes[0].axhline(0, color="black", linestyle="--")
axes[0].set_xlabel("Predicted total_earnings")
axes[0].set_ylabel("Residual (actual - predicted)")
axes[0].set_title(f"Residual Plot — {best_name}")

axes[1].scatter(y_reg_test, best_preds, alpha=0.4, color="steelblue")
lims = [min(y_reg_test.min(), best_preds.min()), max(y_reg_test.max(), best_preds.max())]
axes[1].plot(lims, lims, "r--")
axes[1].set_xlabel("Actual")
axes[1].set_ylabel("Predicted")
axes[1].set_title(f"Predicted vs Actual — {best_name}")

plt.tight_layout()
plt.savefig("reg_best_model_residuals.png", bbox_inches="tight")
plt.show()

# Feature importance for at least one tree-based model (Random Forest, always fitted above)
rf_model_for_importance = fitted_reg_models["Random Forest Regressor"][0]
plt.figure(figsize=(7, 4))
pd.Series(rf_model_for_importance.feature_importances_, index=reg_features).sort_values().plot(kind="barh", color="darkorange")
plt.title("Random Forest Regressor — Feature Importance")
plt.tight_layout()
plt.savefig("reg_rf_feature_importance.png", bbox_inches="tight")
plt.show()

**Observation:** _[Fill in: does the residual plot show a random scatter around zero (good) or a funnel/curve shape (heteroscedasticity, suggesting a missing feature or the need for a log-transformed target)? Which feature dominates importance, and does that match your domain intuition about driver earnings?]_

## Section D — Classification Track, Part A (5 algorithms)

### Member 3 — Classification algorithms 1–5

Target: `burnout_risk` (Low / Medium / High). Each algorithm is trained/evaluated independently with its own confusion-matrix visualisation.

In [ ]:
clf_results = []
fitted_clf_models = {}

def evaluate_classifier(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    acc = accuracy_score(y_test, preds)
    f1w = f1_score(y_test, preds, average="weighted")

    clf_results.append({"Algorithm": name, "Accuracy": acc, "Weighted F1": f1w})
    fitted_clf_models[name] = (model, preds)
    print(f"{name}: Accuracy={acc:.3f}  Weighted F1={f1w:.3f}")

    cm = confusion_matrix(y_test, preds, labels=["Low", "Medium", "High"])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Low", "Medium", "High"])
    fig, ax = plt.subplots(figsize=(5, 5))
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"{name} — Confusion Matrix")
    plt.tight_layout()
    plt.savefig(f"clf_{name.replace(' ', '_').lower()}_confusion_matrix.png", bbox_inches="tight")
    plt.show()

    return model

**1. Logistic Regression** — baseline classifier; interpret coefficients/odds. `# Owner: Member 3`

In [ ]:
# Owner: Member 3
logreg = evaluate_classifier("Logistic Regression", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
                              X_clf_train_s, X_clf_test_s, y_clf_train, y_clf_test)

**2. K-Nearest Neighbors** — tune `k`; distance-based, scaling matters. `# Owner: Member 3`

In [ ]:
# Owner: Member 3
knn_clf_grid = GridSearchCV(KNeighborsClassifier(), param_grid={"n_neighbors": [3, 5, 7, 9, 15]},
                             cv=5, scoring="f1_weighted")
knn_clf_grid.fit(X_clf_train_s, y_clf_train)
print("Best KNN Classifier k:", knn_clf_grid.best_params_)
knn_clf = evaluate_classifier("KNN Classifier",
                               KNeighborsClassifier(n_neighbors=knn_clf_grid.best_params_["n_neighbors"]),
                               X_clf_train_s, X_clf_test_s, y_clf_train, y_clf_test)

**3. Naive Bayes (Gaussian)** — assumes conditional independence between features given the class. `# Owner: Member 3`

In [ ]:
# Owner: Member 3
nb_clf = evaluate_classifier("Gaussian Naive Bayes", GaussianNB(),
                              X_clf_train_s, X_clf_test_s, y_clf_train, y_clf_test)

**4. Decision Tree Classifier** — tune `max_depth`. `# Owner: Member 3`

In [ ]:
# Owner: Member 3
dt_clf_grid = GridSearchCV(DecisionTreeClassifier(random_state=RANDOM_STATE),
                            param_grid={"max_depth": [3, 5, 8, 12, None]}, cv=5, scoring="f1_weighted")
dt_clf_grid.fit(X_clf_train_s, y_clf_train)
print("Best Decision Tree Classifier max_depth:", dt_clf_grid.best_params_)
dt_clf = evaluate_classifier("Decision Tree Classifier",
                              DecisionTreeClassifier(max_depth=dt_clf_grid.best_params_["max_depth"], random_state=RANDOM_STATE),
                              X_clf_train_s, X_clf_test_s, y_clf_train, y_clf_test)

**5. Support Vector Machine (SVC)** — tune `C` and `kernel`; features scaled. `# Owner: Member 3`

In [ ]:
# Owner: Member 3
svc_grid = GridSearchCV(SVC(random_state=RANDOM_STATE), param_grid={"C": [1, 10, 100], "kernel": ["linear", "rbf"]},
                         cv=5, scoring="f1_weighted")
svc_grid.fit(X_clf_train_s, y_clf_train)
print("Best SVC params:", svc_grid.best_params_)
svc_clf = evaluate_classifier("Support Vector Machine",
                               SVC(**svc_grid.best_params_, random_state=RANDOM_STATE),
                               X_clf_train_s, X_clf_test_s, y_clf_train, y_clf_test)

### D2. Preliminary comparison table (Part A)

In [ ]:
clf_comparison = pd.DataFrame(clf_results).sort_values("Weighted F1", ascending=False).reset_index(drop=True)
print(clf_comparison.to_string(index=False))
clf_comparison

## Review 1 — rubric compliance checklist (re-verified against the guidelines)

| Rubric item | Max | Where it's covered |
|---|---|---|
| Primary tools | — | Python 3, scikit-learn, Pandas, NumPy, Matplotlib, Seaborn — all imported/used in cell 2, no other ML library used |
| A1 Dataset loading & audit | 1 | Section A1 — shape, dtypes, missing counts, `trip_total`/categorical distributions |
| A2 EDA visualisations | 2 | Section A2 — 7 numeric distributions, 3 categorical distributions, pickup-hour distribution, correlation heatmap, target distribution, 2 scatter plots |
| A3 Insight commentary | 1 | `**Observation:**` markdown cell after every major visualisation |
| B1 Data cleaning | 1 | Section B1 — justified missing-value strategy, duplicate/outlier handling |
| B2 Encoding, scaling & splitting | 1 | Section B2 — `OneHotEncoder` + `StandardScaler` fit on train only; classification split stratified on `burnout_risk`, regression split stratified on binned `total_earnings` |
| B3 Feature engineering | 1 | Section B3 — `trip_speed_mph`, `pickup_hour`, `is_night`, `is_weekend`, driver-day aggregates, with written justification |
| C1 Regression implementation | 4 | Section C — all 10 algorithms, Members 1 & 2 |
| C2 Comparative evaluation | 2 | Section C2 — `reg_comparison` table, R2/RMSE/MAE, ranked |
| C3 Hyperparameter tuning | 2 | `GridSearchCV` on 8 of 10 models; explicit before/after R2 improvement printed for Ridge and Random Forest |
| C4 Visualisation | 1 | Residual + predicted-vs-actual plot for the **dynamically-selected** actual best model; Random Forest feature importance |
| D1 Classification Part A implementation | 2 | Section D — all 5 algorithms, Member 3 |
| D2 Evaluation | 1 | Accuracy, weighted F1, confusion matrix per algorithm + `clf_comparison` table |
| E1 Presentation quality | 1 | Markdown narrative links every step; ownership tags make individual accountability explicit for the viva |

**Not coverable by a notebook (handle separately, D2/D1 deliverables):**
- **D2 GitHub repository:** create the repo with the required structure (`README.md`, `requirements.txt`, `data/`, `notebooks/`, `models/`, `app/`) and commit incrementally — e.g. one commit per algorithm/section, authored by the actual member who wrote it, so authorship matches the ownership tags above. A single bulk commit gets zero on this criterion regardless of code quality.
- **Viva (5 marks):** not something code can prepare — each member should be ready to explain *any* cell, not just their own algorithms, since guideline 7.4 explicitly says the examiner may question anyone about any part.

**Before your review:** run all cells top-to-bottom with `chicago_taxi_sample.csv` in the same folder as the notebook, fill in every `**Observation:**` markdown cell with your own interpretation, and confirm no cell raises an error.